# MMLU-Pro Neural Network (MLP) with WOE, IV, and RFE

Train on `data/mmlu_pro_full_enriched.csv` using the same pipeline as the XGBoost MMLU-Pro experiment:

- Stratified 70 / 15 / 15 split
- WOE encoding on categoricals
- IV filtering (threshold 0)
- RFE (top 15)
- Simple MLP: 2 hidden layers (64, 32), ReLU, Adam, early stopping, StandardScaler

Script equivalent: `python scripts/train_mlp_mmlu_pro_woe_iv_rfe_complex.py`


In [8]:
import json
from pathlib import Path

import joblib
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "mmlu_pro_full_enriched.csv"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_NAME = "mlp_mmlu_pro_woe_iv_rfe_complex"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15
IV_THRESHOLD = 0
RFE_N_FEATURES = 15
NUMERIC_IV_BINS = 5

HIDDEN_LAYERS = (64, 32)
RFE_HIDDEN_LAYERS = (32,)
MAX_ITER = 200
RFE_MAX_ITER = 40
BATCH_SIZE = 4096
LEARNING_RATE = 0.001
ALPHA = 0.0001
N_ITER_NO_CHANGE = 15
EARLY_STOPPING = True
VALIDATION_FRACTION = 0.1


## Load data


In [9]:
df = pd.read_csv(DATA_PATH)
df["target"] = (df["error"] == "error").astype(int)

print(f"Rows: {len(df):,}")
print(f"Models: {df['llm_model'].nunique()}")
print(df["target"].value_counts())
print()
print(df.groupby(["llm_model", "target"]).size().unstack(fill_value=0).head())
df.head(3)


Rows: 563,787
Models: 47
target
1    282844
0    280943
Name: count, dtype: int64

target                0     1
llm_model                    
DeepSeek-Coder-V2  6586  3765
Llama-2-13b-hf     2830  9202
Llama-2-70b-hf     4366  7666
Llama-2-7b-hf      2207  9825
Meta-Llama-3-70B   6258  5774


,question,llm_model,error,model_name,context_window_tokens,max_output_tokens,vocab_size,positional_encoding_type,attention_type,tokenizer_type,...,question_length_words,question_length_chars,question_complexity_score,has_few_shot_examples,prompt_contains_system_instructions,question_category,is_ambiguous,contains_negation,context_token_count,target
0,"What will be the number of lamps, each having ...",DeepSeek-Coder-V2,no_error,DeepSeek-Coder-V2,131072,8192,102400,RoPE,MoE,bytelevel_BPE,...,47,192,17.74,False,False,Engineering,False,False,58,0
1,Stack is also known as\n\nA. FIFO memory\nB. F...,DeepSeek-Coder-V2,no_error,DeepSeek-Coder-V2,131072,8192,102400,RoPE,MoE,bytelevel_BPE,...,17,84,5.50,False,False,Engineering,False,False,21,0
2,The errors mainly caused by human mistakes are...,DeepSeek-Coder-V2,no_error,DeepSeek-Coder-V2,131072,8192,102400,RoPE,MoE,bytelevel_BPE,...,17,107,11.86,False,False,Engineering,False,False,21,0


## Train / validation / test split


In [10]:
train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["target"],
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=val_ratio,
    random_state=RANDOM_STATE,
    stratify=train_val_df["target"],
)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(train_df["target"].value_counts(normalize=True).round(3))


Train: 394,650 | Val: 84,568 | Test: 84,569
target
1    0.502
0    0.498
Name: proportion, dtype: float64


## Feature engineering and WOE encoding


In [11]:
CATEGORICAL_COLUMNS = [
    "model_name",
    "positional_encoding_type",
    "attention_type",
    "tokenizer_type",
    "question_category",
]

BOOLEAN_COLUMNS = [
    "is_open_source",
    "multilingual_support",
    "has_few_shot_examples",
    "prompt_contains_system_instructions",
    "is_ambiguous",
    "contains_negation",
]

NUMERIC_COLUMNS = [
    "context_window_tokens",
    "max_output_tokens",
    "vocab_size",
    "knowledge_cutoff_year",
    "temperature",
    "top_p",
    "top_k",
    "repetition_penalty",
    "frequency_penalty",
    "presence_penalty",
    "max_tokens_requested",
    "stop_sequences_count",
    "galileo_qa_no_rag",
    "galileo_qa_with_rag",
    "galileo_longform",
    "crag_hallucination_rate",
    "crag_accuracy",
    "question_length_words",
    "question_length_chars",
    "question_complexity_score",
    "context_token_count",
]


def compute_woe_maps(frame, columns, target="target"):
    maps = {}
    total_events = frame[target].sum()
    total_non_events = len(frame) - total_events
    for column in columns:
        grouped = frame.groupby(column, dropna=False)[target].agg(["sum", "count"])
        grouped["non_events"] = grouped["count"] - grouped["sum"]
        woe_map = {}
        for value, row in grouped.iterrows():
            event_rate = (row["sum"] + 0.5) / (total_events + 1.0)
            non_event_rate = (row["non_events"] + 0.5) / (total_non_events + 1.0)
            woe_map[value] = float(np.log(event_rate / non_event_rate))
        maps[column] = woe_map
    return maps


def apply_woe(frame, columns, maps):
    transformed = frame.copy()
    for column in columns:
        transformed[f"{column}_woe"] = transformed[column].map(maps[column]).fillna(0.0)
    return transformed


def build_feature_matrix(frame):
    numeric = frame[NUMERIC_COLUMNS].apply(pd.to_numeric, errors="coerce")
    boolean = frame[BOOLEAN_COLUMNS].astype(int)
    woe_cols = [f"{col}_woe" for col in CATEGORICAL_COLUMNS]
    return pd.concat([numeric, boolean, frame[woe_cols]], axis=1)


woe_maps = compute_woe_maps(train_df, CATEGORICAL_COLUMNS)
train_df = apply_woe(train_df, CATEGORICAL_COLUMNS, woe_maps)
val_df = apply_woe(val_df, CATEGORICAL_COLUMNS, woe_maps)
test_df = apply_woe(test_df, CATEGORICAL_COLUMNS, woe_maps)

feature_names = NUMERIC_COLUMNS + BOOLEAN_COLUMNS + [f"{col}_woe" for col in CATEGORICAL_COLUMNS]

x_train = build_feature_matrix(train_df)
x_val = build_feature_matrix(val_df)
x_test = build_feature_matrix(test_df)

medians = x_train.median(numeric_only=True)
x_train = x_train.fillna(medians)
x_val = x_val.fillna(medians)
x_test = x_test.fillna(medians)

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

print(f"Feature count: {len(feature_names)}")
x_train.head()


Feature count: 32


,context_window_tokens,max_output_tokens,vocab_size,knowledge_cutoff_year,temperature,top_p,top_k,repetition_penalty,frequency_penalty,presence_penalty,...,multilingual_support,has_few_shot_examples,prompt_contains_system_instructions,is_ambiguous,contains_negation,model_name_woe,positional_encoding_type_woe,attention_type_woe,tokenizer_type_woe,question_category_woe
436604,2097152,8192,100277,2024,1.0,1.0,50.0,1.05,0.0,0.0,...,1,0,0,0,0,-0.832226,-0.876366,-0.532957,-0.790557,0.132550
477931,128000,16384,100277,2023,1.0,1.0,50.0,1.05,0.0,0.0,...,1,0,0,0,1,-0.532696,-0.876366,-0.532957,-0.790557,-0.022997
500951,131072,8192,100277,2024,1.0,1.0,50.0,1.05,0.0,0.0,...,1,0,0,0,0,-1.479141,-0.876366,-0.532957,-0.790557,-0.609287
461057,128000,16384,100277,2023,1.0,1.0,50.0,1.05,0.0,0.0,...,1,0,0,0,0,-1.100755,-0.876366,-0.532957,-0.790557,0.132550
319630,131072,8192,100277,2024,1.0,1.0,50.0,1.05,0.0,0.0,...,1,0,0,0,0,-1.577836,-0.876366,-0.532957,-0.790557,-0.013232


## Information Value (IV) filtering


In [12]:
def compute_iv_grouped(frame, group_col, target="target", observed=False):
    events = frame[target].sum()
    non_events = len(frame) - events
    if events == 0 or non_events == 0:
        return 0.0
    grouped = frame.groupby(group_col, dropna=False, observed=observed)[target].agg(["sum", "count"])
    iv = 0.0
    for _, row in grouped.iterrows():
        bad_dist = row["sum"] / events
        good_dist = (row["count"] - row["sum"]) / non_events
        if bad_dist <= 0 or good_dist <= 0:
            continue
        woe = np.log(bad_dist / good_dist)
        iv += (bad_dist - good_dist) * woe
    return float(iv)


def compute_feature_iv(train_frame, feature_name, medians):
    if feature_name.endswith("_woe"):
        raw_col = feature_name.removesuffix("_woe")
        return compute_iv_grouped(train_frame, raw_col)
    if feature_name in BOOLEAN_COLUMNS:
        return compute_iv_grouped(train_frame, feature_name)
    filled = train_frame[feature_name].fillna(medians.get(feature_name, train_frame[feature_name].median()))
    try:
        binned = pd.qcut(filled, q=NUMERIC_IV_BINS, duplicates="drop")
    except ValueError:
        return 0.0
    temp = pd.DataFrame({"bin": binned, "target": train_frame["target"]})
    return compute_iv_grouped(temp, "bin", observed=True)


iv_scores = {name: compute_feature_iv(train_df, name, medians) for name in feature_names}
iv_df = pd.Series(iv_scores, name="iv").sort_values(ascending=False).reset_index()
iv_df.columns = ["feature", "iv"]
iv_df["passes_threshold"] = iv_df["iv"] >= IV_THRESHOLD
iv_df


,feature,iv,passes_threshold
0,model_name_woe,0.566920,True
1,crag_hallucination_rate,0.510831,True
2,crag_accuracy,0.497395,True
3,galileo_qa_no_rag,0.492204,True
4,galileo_qa_with_rag,0.471762,True
5,galileo_longform,0.435571,True
6,top_p,0.373886,True
7,temperature,0.344420,True
8,positional_encoding_type_woe,0.335477,True
9,is_open_source,0.335477,True


In [13]:
iv_features = iv_df.loc[iv_df["passes_threshold"], "feature"].tolist()
if not iv_features:
    iv_features = [iv_df.iloc[0]["feature"]]
print(f"IV selected {len(iv_features)} / {len(feature_names)} features (threshold={IV_THRESHOLD})")
iv_features


IV selected 32 / 32 features (threshold=0)


['model_name_woe',
 'crag_hallucination_rate',
 'crag_accuracy',
 'galileo_qa_no_rag',
 'galileo_qa_with_rag',
 'galileo_longform',
 'top_p',
 'temperature',
 'positional_encoding_type_woe',
 'is_open_source',
 'tokenizer_type_woe',
 'context_window_tokens',
 'max_output_tokens',
 'max_tokens_requested',
 'attention_type_woe',
 'question_category_woe',
 'multilingual_support',
 'vocab_size',
 'question_length_words',
 'question_complexity_score',
 'question_length_chars',
 'context_token_count',
 'contains_negation',
 'repetition_penalty',
 'frequency_penalty',
 'has_few_shot_examples',
 'prompt_contains_system_instructions',
 'is_ambiguous',
 'top_k',
 'knowledge_cutoff_year',
 'presence_penalty',
 'stop_sequences_count']

## Recursive Feature Elimination (RFE)


In [14]:
def build_mlp(hidden_layers, max_iter):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation="relu",
            solver="adam",
            alpha=ALPHA,
            batch_size=BATCH_SIZE,
            learning_rate="adaptive",
            learning_rate_init=LEARNING_RATE,
            max_iter=max_iter,
            early_stopping=EARLY_STOPPING,
            validation_fraction=VALIDATION_FRACTION,
            n_iter_no_change=N_ITER_NO_CHANGE,
            random_state=RANDOM_STATE,
        )),
    ])


def mlp_importance_getter(estimator):
    mlp = estimator.named_steps["mlp"] if hasattr(estimator, "named_steps") else estimator
    return np.abs(mlp.coefs_[0]).mean(axis=1)


n_select = min(RFE_N_FEATURES, len(iv_features))
rfe_estimator = build_mlp(RFE_HIDDEN_LAYERS, RFE_MAX_ITER)
selector = RFE(
    estimator=rfe_estimator,
    n_features_to_select=n_select,
    step=1,
    importance_getter=mlp_importance_getter,
)
selector.fit(x_train[iv_features], y_train)

rfe_ranking = pd.DataFrame({
    "feature": iv_features,
    "ranking": selector.ranking_,
    "selected": selector.support_,
}).sort_values("ranking")

rfe_features = rfe_ranking.loc[rfe_ranking["selected"], "feature"].tolist()
print(f"RFE selected {len(rfe_features)} features")
rfe_ranking


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (

RFE selected 15 features


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


,feature,ranking,selected
0,model_name_woe,1,True
21,context_token_count,1,True
20,question_length_chars,1,True
19,question_complexity_score,1,True
18,question_length_words,1,True
17,vocab_size,1,True
14,attention_type_woe,1,True
9,is_open_source,1,True
15,question_category_woe,1,True
2,crag_accuracy,1,True


## Train Neural Network (MLP) on selected features


In [15]:
x_train_sel = x_train[rfe_features]
x_val_sel = x_val[rfe_features]
x_test_sel = x_test[rfe_features]

model = build_mlp(HIDDEN_LAYERS, MAX_ITER)
model.fit(x_train_sel, y_train)
print(f"Training iterations: {model.named_steps['mlp'].n_iter_}")
model


Training iterations: 50


Pipeline(steps=[('scaler', StandardScaler()),
                ('mlp',
                 MLPClassifier(batch_size=4096, early_stopping=True,
                               hidden_layer_sizes=(64, 32),
                               learning_rate='adaptive', n_iter_no_change=15,
                               random_state=42))])

## Evaluation


In [16]:
def evaluate_split(name, x, y, model):
    proba = model.predict_proba(x)[:, 1]
    preds = (proba >= 0.5).astype(int)
    return {
        "split": name,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, proba),
        "confusion_matrix": confusion_matrix(y, preds),
        "report": classification_report(y, preds, zero_division=0),
    }


scores = pd.DataFrame([
    evaluate_split("train", x_train_sel, y_train, model),
    evaluate_split("val", x_val_sel, y_val, model),
    evaluate_split("test", x_test_sel, y_test, model),
])

scores[["split", "accuracy", "precision", "recall", "f1", "roc_auc"]]


,split,accuracy,precision,recall,f1,roc_auc
0,train,0.688957,0.688269,0.694606,0.691423,0.752647
1,val,0.687541,0.686912,0.693096,0.689990,0.751838
2,test,0.684967,0.684598,0.689891,0.687234,0.747922


In [17]:
for _, row in scores.iterrows():
    print(f"{row['split'].upper()} confusion matrix:\n{row['confusion_matrix']}\n")
    print(row["report"])
    print("-" * 60)


TRAIN confusion matrix:
[[134372  62288]
 [ 60465 137525]]

              precision    recall  f1-score   support

           0       0.69      0.68      0.69    196660
           1       0.69      0.69      0.69    197990

    accuracy                           0.69    394650
   macro avg       0.69      0.69      0.69    394650
weighted avg       0.69      0.69      0.69    394650

------------------------------------------------------------
VAL confusion matrix:
[[28738 13403]
 [13021 29406]]

              precision    recall  f1-score   support

           0       0.69      0.68      0.69     42141
           1       0.69      0.69      0.69     42427

    accuracy                           0.69     84568
   macro avg       0.69      0.69      0.69     84568
weighted avg       0.69      0.69      0.69     84568

------------------------------------------------------------
TEST confusion matrix:
[[28657 13485]
 [13157 29270]]

              precision    recall  f1-score   support



## Inference on test set


In [18]:
test_proba = model.predict_proba(x_test_sel)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

inference_df = test_df[["llm_model", "error", "question_category"]].copy()
inference_df["predicted_error_probability"] = test_proba
inference_df["predicted_label"] = np.where(test_pred == 1, "error", "no_error")
inference_df["correct"] = inference_df["error"] == inference_df["predicted_label"]

print(f"Test accuracy: {inference_df['correct'].mean():.4f}")
print()
print(inference_df.groupby("llm_model")["correct"].mean().sort_values().head(10))
print()
print(inference_df.groupby("question_category")["correct"].mean().sort_values())
inference_df.head(10)


Test accuracy: 0.6850

llm_model
Meta-Llama-3_1-8B-Instruct    0.598982
Meta-Llama-3-70B              0.601314
sonnet_0shots_12_01_18        0.601653
Qwen1.5-72B-Chat              0.602998
deepseek                      0.610043
Meta-Llama-3_1-70B            0.613223
jamba-1.5-large               0.617312
Meta-Llama-3-70B-Instruct     0.619229
flash_0shots_00_35_03         0.619783
mathstral-7B                  0.623779
Name: correct, dtype: float64

question_category
Philosophy          0.624027
History             0.633645
Other               0.645077
Health              0.645122
Computer Science    0.671029
Psychology          0.677801
Economics           0.678559
Business            0.679696
Law                 0.682939
Engineering         0.689759
Physics             0.701276
Biology             0.713770
Math                0.716621
Chemistry           0.733503
Name: correct, dtype: float64


,llm_model,error,question_category,predicted_error_probability,predicted_label,correct
106999,Meta-Llama-3_1-70B,error,Psychology,0.317752,no_error,False
558923,sonnet_0shots_12_01_18,no_error,Economics,0.296078,no_error,True
31918,Llama-2-70b-hf,error,Physics,0.706546,error,True
121607,Meta-Llama-3_1-8B-Instruct,no_error,Math,0.674208,error,False
350743,claude-3-5-haiku-20241022,no_error,Business,0.314598,no_error,True
557364,sonnet_0shots_12_01_18,error,Other,0.374118,no_error,False
194214,Mixtral-8x7B-Instruct-v0.1,no_error,Biology,0.506660,error,False
429844,gemini-1.5-flash-002,no_error,History,0.388000,no_error,True
491737,iask_pro,no_error,Engineering,0.241802,no_error,True
272062,Qwen1.5-7B-Chat,error,Physics,0.747569,error,True


## Feature importance


In [19]:
mlp = model.named_steps["mlp"]
coef_importance = pd.Series(np.abs(mlp.coefs_[0]).mean(axis=1), index=rfe_features).sort_values(ascending=False)
coef_importance


question_category_woe           0.230050
context_token_count             0.188895
question_length_chars           0.175499
question_complexity_score       0.168942
question_length_words           0.162850
model_name_woe                  0.162185
is_open_source                  0.145379
galileo_longform                0.136928
positional_encoding_type_woe    0.135912
crag_accuracy                   0.134216
crag_hallucination_rate         0.133242
attention_type_woe              0.132346
galileo_qa_no_rag               0.132335
vocab_size                      0.128926
top_p                           0.125508
dtype: float64

## Save model and artifacts


In [20]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODEL_DIR / f"{MODEL_NAME}.joblib")

artifact = {
    "feature_names": rfe_features,
    "iv_threshold": IV_THRESHOLD,
    "iv_scores": iv_scores,
    "iv_selected_features": iv_features,
    "rfe_ranking": {row["feature"]: int(row["ranking"]) for _, row in rfe_ranking.iterrows()},
    "woe_maps": {k: {str(key): val for key, val in v.items()} for k, v in woe_maps.items()},
    "numeric_medians": medians.to_dict(),
    "model_params": {
        "hidden_layer_sizes": list(HIDDEN_LAYERS),
        "activation": "relu",
        "solver": "adam",
        "alpha": ALPHA,
        "batch_size": BATCH_SIZE,
        "learning_rate_init": LEARNING_RATE,
        "max_iter": MAX_ITER,
        "early_stopping": EARLY_STOPPING,
        "validation_fraction": VALIDATION_FRACTION,
        "n_iter_no_change": N_ITER_NO_CHANGE,
        "n_iter_": int(model.named_steps["mlp"].n_iter_),
        "loss_": float(model.named_steps["mlp"].loss_),
    },
}
with (MODEL_DIR / f"{MODEL_NAME}_preprocessing.json").open("w", encoding="utf-8") as f:
    json.dump(artifact, f, indent=2)

metrics_payload = {
    "dataset": "mmlu_pro_full_enriched.csv",
    "model_variant": "complex",
    "feature_selection": {
        "total_features": len(feature_names),
        "iv_selected": len(iv_features),
        "rfe_selected": len(rfe_features),
        "selected_features": rfe_features,
    },
    "model_params": artifact["model_params"],
}
for split in ("train", "val", "test"):
    row = scores.loc[scores["split"] == split].iloc[0]
    metrics_payload[split] = {
        "split": split,
        "accuracy": float(row["accuracy"]),
        "precision": float(row["precision"]),
        "recall": float(row["recall"]),
        "f1": float(row["f1"]),
        "roc_auc": float(row["roc_auc"]),
        "confusion_matrix": row["confusion_matrix"].tolist(),
    }
metrics_payload["inference_sample"] = {
    "n_rows": int(len(x_test)),
    "positive_predictions": int(test_pred.sum()),
    "mean_predicted_probability": float(test_proba.mean()),
}

with (MODEL_DIR / f"{MODEL_NAME}_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

print(f"Saved model to {MODEL_DIR / f'{MODEL_NAME}.joblib'}")


Saved model to /Users/konstantine25b/Desktop/Gaia Student Club/Retrival Failure/models/mlp_mmlu_pro_woe_iv_rfe_complex.joblib
